# ODI to Databricks Migration

**Source File:** `HR.TRG_EMP_INSERT.sql`
**Conversion Timestamp:** 2024-07-30T12:00:00Z
**Description:** This notebook loads data into the `TRG_EMP` table from `EMPLOYEES`.

In [ ]:
dbutils.widgets.text("ETL_JOB_TYPE", "FULL_LOAD", "1. ETL Job Type (e.g., FULL_LOAD, INCREMENTAL)")
dbutils.widgets.text("DATASOURCE_NUM_ID", "0", "2. Data Source Number ID")
dbutils.widgets.text("ETL_PROC_WID", "0", "3. ETL Process Widget ID")
dbutils.widgets.text("ODI_SESS_NO", "0", "4. ODI Session Number")
dbutils.widgets.text("ETL_LAST_EXTRACT_TIME", "1900-01-01 00:00:00", "5. Last Extract Timestamp (YYYY-MM-DD HH:mm:ss)")
dbutils.widgets.text("ETL_CURRENT_EXTRACT_TIME", "9999-12-31 23:59:59", "6. Current Extract Timestamp (YYYY-MM-DD HH:mm:ss)")

# ETL Parameters

In [ ]:
%sql
-- Create temporary views for ETL parameters
CREATE OR REPLACE TEMPORARY VIEW v_etl_job_type AS
SELECT '${ETL_JOB_TYPE}' AS etl_job_type;

CREATE OR REPLACE TEMPORARY VIEW v_datasource_num_id AS
SELECT ${DATASOURCE_NUM_ID} AS datasource_num_id;

CREATE OR REPLACE TEMPORARY VIEW v_etl_proc_wid AS
SELECT ${ETL_PROC_WID} AS etl_proc_wid;

CREATE OR REPLACE TEMPORARY VIEW v_odi_sess_no AS
SELECT ${ODI_SESS_NO} AS odi_sess_no;

CREATE OR REPLACE TEMPORARY VIEW v_etl_last_extract_time AS
SELECT to_timestamp('${ETL_LAST_EXTRACT_TIME}', 'yyyy-MM-dd HH:mm:ss') AS etl_last_extract_time;

CREATE OR REPLACE TEMPORARY VIEW v_etl_current_extract_time AS
SELECT to_timestamp('${ETL_CURRENT_EXTRACT_TIME}', 'yyyy-MM-dd HH:mm:ss') AS etl_current_extract_time;

In [ ]:
print(f"ETL_JOB_TYPE: {dbutils.widgets.get('ETL_JOB_TYPE')}")
print(f"DATASOURCE_NUM_ID: {dbutils.widgets.get('DATASOURCE_NUM_ID')}")
print(f"ETL_PROC_WID: {dbutils.widgets.get('ETL_PROC_WID')}")
print(f"ODI_SESS_NO: {dbutils.widgets.get('ODI_SESS_NO')}")
print(f"ETL_LAST_EXTRACT_TIME: {dbutils.widgets.get('ETL_LAST_EXTRACT_TIME')}")
print(f"ETL_CURRENT_EXTRACT_TIME: {dbutils.widgets.get('ETL_CURRENT_EXTRACT_TIME')}")

# Data Loading

In [ ]:
%sql
-- SCEN_TASK_NO in {10}
-- SCEN_TASK_NO in {20}

-- SCEN_TASK_NO in {30}: Insert into target table HR.TRG_EMP
INSERT INTO workspace.hr.trg_emp
(
  EMPLOYEE_ID ,
  FIRST_NAME ,
  LAST_NAME ,
  EMAIL ,
  PHONE_NUMBER ,
  HIRE_DATE ,
  JOB_ID ,
  SALARY ,
  COMMISSION_PCT ,
  MANAGER_ID ,
  DEPARTMENT_ID
)
SELECT
  EMPLOYEES.EMPLOYEE_ID ,
  EMPLOYEES.FIRST_NAME ,
  EMPLOYEES.LAST_NAME ,
  EMPLOYEES.EMAIL ,
  EMPLOYEES.PHONE_NUMBER ,
  EMPLOYEES.HIRE_DATE ,
  EMPLOYEES.JOB_ID ,
  EMPLOYEES.SALARY ,
  EMPLOYEES.COMMISSION_PCT ,
  EMPLOYEES.MANAGER_ID ,
  EMPLOYEES.DEPARTMENT_ID
FROM
  workspace.hr.employees AS EMPLOYEES;

# Validation

In [ ]:
%sql
SELECT COUNT(*) AS total_records_in_trg_emp
FROM workspace.hr.trg_emp;

# Conversion Notes

**Manual Actions Required:**

1.  **Target Table DDL:** The original ODI snippet did not include `CREATE TABLE` DDL for `HR.TRG_EMP`. Ensure that `workspace.hr.trg_emp` has been created with the appropriate Databricks Spark SQL data types (e.g., `VARCHAR2` -> `STRING`, `NUMBER` -> `BIGINT`/`DECIMAL`, `DATE` -> `TIMESTAMP`).
2.  **Source Table Existence:** Verify that `workspace.hr.employees` exists and contains the expected data.
3.  **Widget Values:** Adjust default widget values (`ETL_JOB_TYPE`, `DATASOURCE_NUM_ID`, etc.) or pass them via notebook jobs if required by your ETL framework.
4.  **Empty SCEN_TASK_NOs:** The original file contained empty `SCEN_TASK_NO` blocks ({10} and {20}) which are treated as no-op session markers and are preserved as comments in the adjacent SQL cell.